# Recurrent Neural Networks and Sequence Modeling

In this notebook, we study neural networks for sequential data.

Unlike images or tabular data, sequences have an order. The meaning of an element often depends on what came before it. Examples include text, speech, video, motion, weather data, and stock-market time series.

We will focus on:

1. why sequence modeling needs memory
2. how words or characters can be represented numerically
3. how recurrent neural networks process sequences step by step
4. how to train a character-level language model
5. how sampling and temperature affect generated text
6. why vanilla RNNs suffer from vanishing and exploding gradients
7. how LSTM and GRU models address some of these problems


## 1. Why Sequence Modeling?

Many neural networks we have used so far process fixed-size inputs and produce fixed-size outputs.

For example, an image classifier receives one image and predicts one class.

Sequences are different:

- they can have variable length
- the order of elements matters
- predictions often require previous context
- the same operation should be applied at different time steps

For example:

> I love neural ...

A good model should use the previous words to predict that a likely next word could be:

> networks

In another example:

> France is where I grew up, but now I live in Germany. I speak fluent ...

The correct prediction depends on context from much earlier in the sentence.

This is called a **long-term dependency**.

## 2. Representing Words and Characters

Neural networks cannot process raw text directly.
We first need to convert words or characters into numerical representations.

A simple approach is:

1. build a vocabulary
2. assign each unique token an integer index
3. convert text into a sequence of indices

For example:

```text
"hello"
```

could become:
```text
[1, 0, 2, 2, 3]
```
if the vocabulary is:
```text
{"e": 0, "h": 1, "l": 2, "o": 3}
```
Later, these indices can be converted into one-hot vectors or dense embeddings.

## Task 1: Build a Character Vocabulary

In [ ]:
text = "hello recurrent neural networks"

# TODO: get sorted unique characters
chars = ...

# TODO: create character-to-index dictionary
char_to_idx = ...

# TODO: create index-to-character dictionary
idx_to_char = ...

print("Vocabulary:", chars)
print("Vocabulary size:", len(chars))
print("Character to index:", char_to_idx)

## Task 2: Encode and Decode Text

In [ ]:
def encode_text(text, char_to_idx):
    """
    Convert a text string into a list of integer indices.
    """

    # TODO: encode each character
    encoded = ...

    return encoded


def decode_text(indices, idx_to_char):
    """
    Convert a list of integer indices back into text.
    """

    # TODO: decode each index
    decoded = ...

    return decoded

In [ ]:
encoded = encode_text(text, char_to_idx)
decoded = decode_text(encoded, idx_to_char)

print("Original text:", text)
print("Encoded text:", encoded)
print("Decoded text:", decoded)

### One-Hot Encoding

One-hot encoding represents each token as a vector with length equal to the vocabulary size.

Only one position is active:

```text
[0, 0, 1, 0, 0]
```
This works, but it has two problems:

* large vocabularies create very large vectors
* all tokens are equally distant from each other

For example, one-hot vectors do not know that love and like are semantically closer than love and car.

### Dense Embeddings

Instead of large one-hot vectors, neural networks usually learn dense embeddings.

An embedding maps each token index to a smaller trainable vector.

For example, a vocabulary of size 10000 could use embeddings of size 300 instead of one-hot vectors of size 10000.

In [ ]:
# Example
import torch

vocab_size = len(chars)
embedding_dim = 8

embedding_layer = torch.nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

encoded_tensor = torch.tensor(encoded)

embedded = embedding_layer(encoded_tensor)

print("Input indices shape:", encoded_tensor.shape)
print("Embedded sequence shape:", embedded.shape)

**Questions**

1. Why can neural networks not directly process raw text?
2. What is one disadvantage of one-hot encoding?
3. What does an embedding layer learn?
4. Why are dense embeddings more useful than one-hot vectors?

## 3. Preparing Sequential Data

For sequence prediction, the model usually receives part of a sequence and learns to predict the next element.

Example:

```text
Text:    h e l l o
Input:   h e l l
Target:  e l l o
```

At every time step, the model sees the current character and should predict the next character.

This is the basic setup for a character-level language model.

## Task 3: Create Input and Target Sequences

In [ ]:
sample_text = "hello"

encoded_sample = encode_text(sample_text, char_to_idx)

# TODO: create input sequence
input_sequence = ...

# TODO: create target sequence
target_sequence = ...

print("Text:", sample_text)
print("Encoded:", encoded_sample)
print("Input sequence:", input_sequence)
print("Target sequence:", target_sequence)

For longer text, we can create many training examples using a fixed sequence length.

For example, with sequence length 4:

```text
"hello recurrent"
```
could produce examples like:

```
Input:  hell   Target: ello
Input:  ello   Target: llo
Input:  llo    Target: lo r
```

## Task 4: Create Many Training Sequences

In [ ]:
def create_sequences(encoded_text, sequence_length):
    """
    Create input-target pairs for next-character prediction.

    Args:
        encoded_text: list of integer character indices
        sequence_length: number of characters in each input sequence

    Returns:
        inputs: tensor of shape (num_sequences, sequence_length)
        targets: tensor of shape (num_sequences, sequence_length)
    """

    inputs = []
    targets = []

    for i in range(len(encoded_text) - sequence_length):
        # TODO: create one input sequence
        input_seq = ...

        # TODO: create the corresponding target sequence
        target_seq = ...

        inputs.append(input_seq)
        targets.append(target_seq)

    inputs = torch.tensor(inputs, dtype=torch.long)
    targets = torch.tensor(targets, dtype=torch.long)

    return inputs, targets

In [ ]:
sequence_length = 8

inputs, targets = create_sequences(encoded, sequence_length)

print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)

print("First input:", decode_text(inputs[0].tolist(), idx_to_char))
print("First target:", decode_text(targets[0].tolist(), idx_to_char))

**Questions**

1. Why is the target sequence shifted by one character?
2. Why do we use many short sequences instead of feeding the whole text at once?
3. What does `sequence_length` control?

## 4. Recurrent Computation Step by Step

A recurrent neural network processes a sequence one element at a time.

At each time step, it combines:

- the current input \(x_t\)
- the previous hidden state \(h_{t-1}\)

to compute a new hidden state:

$$
h_t = \tanh(W_x x_t + W_h h_{t-1} + b)
$$

The same weights are reused at every time step.
This is called **weight sharing across time**.

## Task 5: Implement a Simple Forward Pass

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# Use one sequence from the previous section
input_seq = inputs[0]

embedding_dim = 8
hidden_size = 16

embedding = nn.Embedding(vocab_size, embedding_dim)

Wx = nn.Linear(embedding_dim, hidden_size)
Wh = nn.Linear(hidden_size, hidden_size, bias=False)
Wy = nn.Linear(hidden_size, vocab_size)

In [ ]:
def simple_rnn_forward(input_seq):
    """
    Process one input sequence step by step.

    Args:
        input_seq: tensor of shape (sequence_length,)

    Returns:
        outputs: tensor of shape (sequence_length, vocab_size)
        hidden_states: tensor of shape (sequence_length, hidden_size)
    """

    h = torch.zeros(hidden_size)

    outputs = []
    hidden_states = []

    for token_idx in input_seq:
        # Convert token index to embedding vector
        x = embedding(token_idx)

        # TODO: compute new hidden state
        h = ...

        # TODO: compute output logits
        y = ...

        outputs.append(y)
        hidden_states.append(h)

    outputs = torch.stack(outputs)
    hidden_states = torch.stack(hidden_states)

    return outputs, hidden_states

In [ ]:
outputs, hidden_states = simple_rnn_forward(input_seq)

print("Input sequence shape:", input_seq.shape)
print("Outputs shape:", outputs.shape)
print("Hidden states shape:", hidden_states.shape)

**Questions**

1. Why is the hidden state initialized before the loop?
2. Why is the same `Wx`, `Wh`, and `Wy` used at every time step?
3. What information can the hidden state contain?

## 5. Character-Level RNN in PyTorch

Now we build a character-level language model.

The model receives a sequence of character indices and predicts the next character at each time step.

The model has three parts:

1. **Embedding layer**
   Converts character indices into dense vectors.

2. **RNN layer**
   Processes the sequence step by step and updates the hidden state.

3. **Linear output layer**
   Converts hidden states into logits over the vocabulary.

## Task 6: Define a Character-Level RNN

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        # TODO: define embedding layer
        self.embedding = ...

        # TODO: define RNN layer
        self.rnn = ...

        # TODO: define output layer
        self.fc = ...

    def forward(self, x, hidden=None):
        # x shape: (batch_size, sequence_length)

        # TODO: convert indices to embeddings
        x = ...

        # TODO: pass embeddings through RNN
        output, hidden = ...

        # TODO: convert RNN outputs to vocabulary logits
        logits = ...

        return logits, hidden

In [ ]:
embedding_dim = 16
hidden_size = 32

model = CharRNN(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size
)

batch = inputs[:4]

logits, hidden = model(batch)

print("Input batch shape:", batch.shape)
print("Logits shape:", logits.shape)
print("Hidden shape:", hidden.shape)

**Questions**

1. Why does the model output one prediction per time step?
2. What does `batch_first=True` change?
3. Why is the final layer output size equal to `vocab_size`?

## 6. Training the Character-Level RNN

We now train the RNN to predict the next character at every time step.

For each input sequence, the target sequence is shifted by one character.

Example:

```text
Input:  recur
Target: ecurr
```
The model outputs logits with shape:
```text
(batch_size, sequence_length, vocabulary_size)
```
For CrossEntropyLoss, we reshape this into:
```text
(batch_size * sequence_length, vocabulary_size)
```

In [ ]:
# Use a slightly larger text:

training_text = """
recurrent neural networks process sequences.
sequences have order and context.
the hidden state stores information from previous time steps.
recurrent models share weights across time.

a sequence can be short or long.
some predictions need only recent context.
other predictions need information from many steps before.
language models predict the next symbol from previous symbols.
rnn models update their hidden state at every time step.
the same weights are reused for each position in the sequence.
"""

training_text = training_text.lower() * 5

In [ ]:
# Rebuild vocabulary data
chars = sorted(list(set(training_text)))

char_to_idx = {char: idx for idx, char in enumerate(chars)}
idx_to_char = {idx: char for idx, char in enumerate(chars)}

vocab_size = len(chars)

encoded = encode_text(training_text, char_to_idx)

sequence_length = 40
inputs, targets = create_sequences(encoded, sequence_length)

print("Vocabulary size:", vocab_size)
print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)

In [ ]:
# Create dataloader
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(inputs, targets)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
# Model setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_dim = 32
hidden_size = 64

model = CharRNN(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

## Task 7: Implement one epoch training loop

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    total_loss = 0

    for input_batch, target_batch in dataloader:
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)

        # TODO: reset gradients
        ...

        # TODO: forward pass
        logits, hidden = ...

        # TODO: reshape logits and targets for CrossEntropyLoss
        logits = ...
        target_batch = ...

        # TODO: compute loss
        loss = ...

        # TODO: backward pass
        ...

        # TODO: update parameters
        ...

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    return avg_loss

In [ ]:
# Training loop
num_epochs = 10

losses = []

for epoch in range(num_epochs):
    loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    losses.append(loss)

    print(f"Epoch {epoch + 1}/{num_epochs} | Loss: {loss:.4f}")

In [ ]:
# Plot loss
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

**Questions**

1. Why do we reshape the logits before applying `CrossEntropyLoss`?
2. Why is the target sequence shifted by one character?
3. What should happen to the loss if training works?

## 7. Generating Text and Temperature

After training, the RNN can generate text one character at a time.

At each step:

1. give the model a starting text
2. predict logits for the next character
3. convert logits into probabilities with softmax
4. sample the next character
5. append it to the generated text

The **temperature** controls how random the sampling is:

- low temperature: more conservative, repetitive text
- high temperature: more diverse, but more mistakes

## Task 8: Implement Text Generation

In [ ]:
def generate_text(model, start_text, length, temperature, char_to_idx, idx_to_char, device):
    """
    Generate text using a trained character-level RNN.

    Args:
        model: trained CharRNN model
        start_text: initial text prompt
        length: number of characters to generate
        temperature: controls randomness
        char_to_idx: character-to-index dictionary
        idx_to_char: index-to-character dictionary
        device: torch device

    Returns:
        generated text string
    """

    model.eval()

    generated = start_text.lower()

    for _ in range(length):
        # Use only recent context
        context = generated[-sequence_length:]

        input_indices = encode_text(context, char_to_idx)
        input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)

        with torch.no_grad():
            logits, hidden = model(input_tensor)

        # TODO: get logits for the last time step
        next_char_logits = ...

        # TODO: apply temperature
        next_char_logits = ...

        # TODO: convert logits to probabilities
        probabilities = ...

        # TODO: sample next character index
        next_char_idx = ...

        # TODO: convert index to character
        next_char = ...

        generated += next_char

    return generated

In [ ]:
torch.manual_seed(42)

untrained_model = CharRNN(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size
).to(device)

print("Generated text before training:")
print(generate_text(
    model=untrained_model,
    start_text="recurrent ",
    length=200,
    temperature=0.8,
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    device=device
))

In [ ]:
start_text = "the "

for temperature in [0.3, 0.9, 1.5]:
    generated = generate_text(
        model=model,
        start_text=start_text,
        length=200,
        temperature=temperature,
        char_to_idx=char_to_idx,
        idx_to_char=idx_to_char,
        device=device
    )

    print(f"\nTemperature: {temperature}")
    print(generated)

In the trained RNN, temperature may be less dramatic because the model has learned a small repeated corpus very well. Still, high temperature usually introduces more mistakes and unusual character combinations.

**Questions**

1. What happens when the temperature is very low?
2. What happens when the temperature is very high?
3. Why do we sample from the probability distribution instead of always choosing the most likely character?

## 8. Vanishing and Exploding Gradients

RNNs are trained by unfolding them through time and applying backpropagation through the full sequence.

This is called **Backpropagation Through Time**.

A problem with vanilla RNNs is that gradients are repeatedly multiplied through many time steps.

If gradients become very small, the model struggles to learn long-term dependencies.
This is called the **vanishing gradient problem**.

If gradients become very large, training can become unstable.
This is called the **exploding gradient problem**.

One common solution for exploding gradients is **gradient clipping**.

Gradient clipping rescales gradients when their norm becomes too large.

## Task 9: Add Gradient Clipping

In [ ]:
def train_one_epoch_with_clipping(model, dataloader, criterion, optimizer, device, max_norm):
    model.train()

    total_loss = 0

    for input_batch, target_batch in dataloader:
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device)

        optimizer.zero_grad()

        logits, hidden = model(input_batch)

        logits = logits.reshape(-1, vocab_size)
        target_batch = target_batch.reshape(-1)

        loss = criterion(logits, target_batch)

        loss.backward()

        # TODO: clip gradients before updating parameters
        ...

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    return avg_loss

In [ ]:
# Training loop:
num_epochs = 10
max_norm = 1.0

clipped_losses = []

for epoch in range(num_epochs):
    loss = train_one_epoch_with_clipping(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        max_norm
    )

    clipped_losses.append(loss)

    print(f"Epoch {epoch + 1}/{num_epochs} | Loss: {loss:.4f}")

In [ ]:
# Plotting
plt.figure(figsize=(8, 4))
plt.plot(clipped_losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss with Gradient Clipping")
plt.show()

In this small example, gradient clipping may not noticeably improve the loss because the model is already stable. The purpose here is to show where clipping is applied in the training loop.

**Questions**

1. Why are RNNs especially affected by vanishing and exploding gradients?
2. What does gradient clipping prevent?
3. Does gradient clipping solve vanishing gradients?

## 9. LSTM and GRU

Vanilla RNNs can struggle with long-term dependencies because information has to pass through many repeated hidden-state updates.

LSTMs and GRUs are gated recurrent networks designed to handle this problem better.

An **LSTM** has an additional cell state \(c_t\), which can carry information across many time steps. Gates control what should be kept, forgotten, or used as output.

A **GRU** is a simpler gated recurrent model. It has fewer parameters than an LSTM because it does not use a separate cell state.

In this section, we replace the vanilla RNN with an LSTM.

## Task 10: Replace the RNN with an LSTM

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.hidden_size = hidden_size

        # TODO: define embedding layer
        self.embedding = ...

        # TODO: define LSTM layer
        self.lstm = ...

        # TODO: define output layer
        self.fc = ...

    def forward(self, x, hidden=None):
        # x shape: (batch_size, sequence_length)

        # TODO: convert character indices to embeddings
        x = ...

        # TODO: pass embeddings through LSTM
        output, hidden = ...

        # TODO: convert LSTM outputs to vocabulary logits
        logits = ...

        return logits, hidden

In [ ]:
lstm_model = CharLSTM(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size
).to(device)

batch = inputs[:4].to(device)

logits, hidden = lstm_model(batch)

print("Input batch shape:", batch.shape)
print("Logits shape:", logits.shape)
print("Hidden state shape:", hidden[0].shape)
print("Cell state shape:", hidden[1].shape)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.003)

num_epochs = 10
max_norm = 1.0

lstm_losses = []

for epoch in range(num_epochs):
    loss = train_one_epoch_with_clipping(
        lstm_model,
        train_loader,
        criterion,
        optimizer,
        device,
        max_norm
    )

    lstm_losses.append(loss)

    print(f"Epoch {epoch + 1}/{num_epochs} | LSTM Loss: {loss:.4f}")

On this small repeated text, the LSTM may not perform much better than the vanilla RNN. The advantage of LSTMs becomes clearer for longer sequences and tasks with stronger long-term dependencies.

In [ ]:
generated = generate_text(
    model=lstm_model,
    start_text="recurrent ",
    length=200,
    temperature=0.8,
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    device=device
)

print(generated)

In [ ]:
# Optionally you can compare with GRU
class CharGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)

        output, hidden = self.gru(x, hidden)

        logits = self.fc(output)

        return logits, hidden

In [ ]:
gru_model = CharGRU(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size
).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("RNN parameters:", count_parameters(model))
print("LSTM parameters:", count_parameters(lstm_model))
print("GRU parameters:", count_parameters(gru_model))

The GRU usually has fewer parameters than the LSTM because it uses a simpler gating mechanism and does not maintain a separate cell state.

**Questions**

1. What additional state does an LSTM have compared to a vanilla RNN?
2. Why can LSTMs help with long-term dependencies?
3. Why does a GRU usually have fewer parameters than an LSTM?
4. In the printed LSTM output, what is the difference between hidden state and cell state?